In [ ]:
import torch
import numpy as np
import torch.nn as nn
import helper
from transformers import BertTokenizer, BertModel, BertForSequenceClassification


In [ ]:
# creates the bert tokenizer object
# generates the attengion masks.
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

#loads the pretrained BERT-base weights
#wraps berte with a clasification head
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
# softmax -> -logptrueclass -> av. over batch
criterion = nn.CrossEntropyLoss()


# bet uses the MRPC Microsoft Research Paraphrase Corpus
# batch_text_A, list of sentences sent1 in MRPC, batch_text_Be-> sent2 in MRPC, batch_labels-> 0/1 binary tasks
def loss_on_batch(texts_a, texts_b, labels):
  #tokenizator, convert raw text-> tensors
    enc = tokenizer(texts_a, texts_b, padding=True, truncation=True, return_tensors="pt")
    logits = model(**enc).logits  # shape: [batch_size, K]
    labels = torch.tensor(labels) # shape: [batch_size], values in {0,...,K-1}
    loss = criterion(logits, labels)
    return loss


In [ ]:
Bmodel.bert.encoder.layer
for i, layer in enumerate(model.bert.encoder.layer):
    print(f"--- Layer {i} ---")
    print(layer)


Stage 1: Generate S pruning strategies, containing varying pruning ratios strategies. Bj= (B0^j, B1^j, B2^j,... BN^j), j = 1,...,S and N=12, encoder layers of BERT.

Stage 1: Weight Prunning, M is the Magnitude Mask.

Evaluate sub-net, no finetuning, using quick evaluation

In [ ]:
def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    model.to(device)

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attn, labels=labels)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

        correct += (preds == labels).sum().item()
        total += input_ids.size(0)

        # Fast evaluation: break early
        if total >= 500:
            break

    return correct / total


In [ ]:
def ae_bert(model, dataloader, n_candidates=20, target_p=0.7, device="cuda"):
    num_layers = len(model.bert.encoder.layer) # those are 12

    best_score = -1
    best_model = None

    for i in range(n_candidates):
        print(f"Candidate {i+1}/{n_candidates}")

        # 1. generate pruning strategy
        strategy = generate_strategy(num_layers, target_p)

        # 2. apply pruning
        pruned_model = apply_pruning(model, strategy)

        # 3. evaluate
        score = evaluate_model(pruned_model, dataloader, device)

        print(f" → score = {score:.4f}")

        if score > best_score:
            best_score = score
            best_model = pruned_model

    return best_model


In [ ]:
from datasets import load_dataset
dataset = load_dataset("glue", "mrpc")
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

#to change to QNLI, RTE batch["sentence"], batch["question"]
def encode_batch(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

encoded = dataset.map(encode_batch, batched=True)
encoded = encoded.rename_column("label", "labels")
encoded.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


# Build the Quick evaluation dataloader
from torch.utils.data import DataLoader, Subset
import torch
import numpy as np

def get_quick_dataloader(encoded_dataset, batch_size=32, n_samples=500):

    idx = np.random.choice(len(encoded_dataset["train"]), n_samples, replace=False)
    subset = Subset(encoded_dataset["train"], idx)

    dataloader = DataLoader(subset, batch_size=batch_size, shuffle=False)
    return dataloader

quick_train_loader = get_quick_dataloader(encoded)
#score = evaluate_model(pruned_model, quick_train_loader, device)

#Full dataloaer for training/fine-tunning
full_train_loader = DataLoader(encoded["train"], batch_size=32, shuffle=True)
full_val_loader   = DataLoader(encoded["validation"], batch_size=32)


In [ ]:
# Load base model
import importlib
import helper

importlib.reload(helper)
from helper import generate_strategy, apply_pruning
from transformers import TrainingArguments

model = BertForSequenceClassification.from_pretrained("bert-base-uncased")

# Stage I/II — find best candidate
best_candidate = ae_bert(
    model,
    dataloader=quick_train_loader,
    n_candidates=3,
    target_p=0.3,
    device="cuda"
)


In [ ]:
from helper import finetune
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    )

model_finetuned = finetune(best_candidate, full_train_loader, full_val_loader, args)